<a href="https://colab.research.google.com/github/yuhui-0611/ESAA/blob/main/ESAA_OB_WEEK04_2_Code_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KRX 주식 투자 알고리즘 경진대회

[link text](https://dacon.io/competitions/official/236117/codeshare/8722?page=1&dtype=recent)

## 코드 흐름 요약

- 주식 종목별 과거 종가 데이터를 이용해 다음 날 종가를 예측
- 그 예측값을 다시 입력 데이터에 붙여 가면서 향후 15거래일의 흐름을 순차적으로 예측
- 최종적으로 종목별 점수를 계산하여 순위를 매기는 구조


1. 라이브러리 불러오기 및 시드 고정
    - seed_everything() 함수를 정의해서 random seed를 고정
2. 날짜 및 파생변수 생성 함수 정의
    - make_month(df) : 일자에서 월 추출
    - make_year(df) : 일자에서 연도 추출
    - make_day(df) : 일자에서 일(day) 추출
    - make_weekday(df) : 요일 추출

3. 종목별 Feature Engineering
    - Feature_Engineering(df, code_name) 함수에서 특정 종목코드에 해당하는 데이터만 추출
      - 월, 연도, 일, 요일 변수 생성
      - target4 = 종가.shift(-1) 생성 → 오늘 데이터를 이용해 다음 날 종가를 맞추기 위한 타깃 변수
      - 마지막 결측치는 bfill로 채움
      - 시가, 고가, 저가는 제거

4. 시차 변수 생성
    - make_shift(df, n_days) 함수 : 종가에 대해 과거 n일치 값을 feature로 추가
    > 주가 데이터는 현재 값이 과거 값의 영향을 강하게 받기 때문에, 과거 10일간의 종가 흐름을 입력 변수로 활용


```
tmp_df = Feature_Engineering(krs_Df, code)
make_shift(tmp_df, 10)
```



5. 종목별 모델 학습, CatBoostRegressor 학습


```
target4 = catboost.CatBoostRegressor(random_state=138, verbose=2000, iterations=1500)
target4.fit(train_x, train_y['target4'])
```


6. 15일 미래 가격을 순차 예측

```
for idx in range(15):
    target4_pred = target4.predict(valid_x)
```

- 예측한 값을 다시 다음 입력 데이터에 붙여 넣어 연쇄적으로 미래를 예측
    > recursive forecasting(재귀적 다중시점 예측) 방식

7. 종목별 score 계산
    - 예측된 종가 흐름을 바탕으로 다음 날 대비 상승률을 누적


## 새롭게 알게 된 내용

- 주가 예측에서 단순히 오늘 종가만 쓰는 것이 아니라 날짜 파생변수와 과거 시차 변수(lag feature)를 함께 사용한다는 것
- 특히 shift_종가_1 ~ shift_종가_10처럼 과거 여러 시점의 종가를 feature로 넣는 방식은 시계열 데이터 처리에서 매우 자주 사용되는 핵심 기법
- CatBoost가 분류만이 아니라 회귀 문제에도 충분히 활용 가능하다는 점

## 배울 점

- 모든 종목을 하나의 모델로 처리하지 않고, 종목마다 따로 모델을 학습한 점
> 주식은 종목마다 움직임 특성이 다르기 때문
- 시계열 문제를 전통적인 ARIMA 같은 방식이 아니라, 일반 머신러닝 회귀 문제처럼 feature engineering해서 푼다는 점
  - 날짜 변수 만들기, lag 변수 만들기, 다음 날 가격을 target으로 설정하기
  - 이 과정을 통해 시계열 문제를 지도학습 문제로 바꾸는 사고방식을 배움